# 03 · Cross-lingual retrieval and the RAG chatbot
The policy is English; questions are English, Tagalog or Taglish. This notebook measures which retrieval signals find the right page, then runs the full chatbot.

In [ ]:
import sys, json
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))
import pandas as pd, numpy as np
from src import config
pd.set_option("display.max_colwidth", 120)
from src.retriever import HybridRetriever, bank_frame
from src.glossary import expand_query
qa = pd.read_csv(config.QA_FILE)

## 1. Glossary expansion

In [ ]:
for q in ["Nawala ng airline ang bagahe ko", "Na-delay yung flight ko ng 10 oras", "Sagot ba kung buntis ako?"]:
    print(q, "\n   ->", expand_query(q))

## 2. Ablation: Hit@k of the gold page (unseen test questions)
Bank = training items only.

In [ ]:
CONFIGS = {"tfidf": dict(lexical="tfidf", use_glossary=False, use_bank=False),
           "bm25": dict(lexical="bm25", use_glossary=False, use_bank=False),
           "bm25+glossary": dict(lexical="bm25", use_glossary=True, use_bank=False),
           "bm25+bank": dict(lexical="bm25", use_glossary=False, use_bank=True),
           "full": dict(lexical="bm25", use_glossary=True, use_bank=True)}
test = qa[qa.split == "test"]
bank = bank_frame(splits=("train",))
rows = []
for name, kw in CONFIGS.items():
    ret = HybridRetriever(qa=bank, **kw)
    for r in test.itertuples():
        gold = {int(p) for p in str(r.pages).split(";") if p not in ("", "nan")}
        if gold:
            top = ret.top_pages(r.input, 5)
            rows.append({"config": name, "language": r.language, "hit@1": bool(gold & set(top[:1])),
                         "hit@3": bool(gold & set(top[:3])), "hit@5": bool(gold & set(top))})
res = pd.DataFrame(rows)
res.groupby(["config", "language"])["hit@3"].mean().unstack().loc[list(CONFIGS)].round(2)

In [ ]:
res.groupby("config")[["hit@1", "hit@3", "hit@5"]].mean().loc[list(CONFIGS)].round(2)

The glossary is what rescues Tagalog questions (Hit@3 0.56 → 0.89). The full results, including the FAQ-paraphrase scenario, are produced by `scripts/evaluate.py` → `results/metrics.json` and `results/charts/`.

## 3. The chatbot
Without `ANTHROPIC_API_KEY` it runs in offline mode (checked reference answers). With a key, Claude writes a grounded answer citing *[Section, p.]* in the customer's language.

In [ ]:
from src.chatbot import PolicyChatbot
bot = PolicyChatbot()
for q in ["Nawala ng airline yung maleta ko, ano pwede ko i-claim?",
          "My flight was delayed 10 hours, what can I get?",
          "Sagot ba ng insurance kung buntis ako at nanganak sa Japan?",
          "Ano ang magandang kainan sa Tokyo?"]:
    r = bot.answer(q)
    print(f"Q: {q}\n[{r.mode} | {r.language} | {r.topic} {r.topic_confidence:.0%} | pages {[c['page'] for c in r.chunks]}]\n{r.answer}\n")

In [ ]:
# what Claude actually receives (system prompt + grounded user turn)
p = bot.prepare("Covered ba ako pag nag-scuba diving ako sa Palawan?")
print(p["system"][:900], "...\n")
print(p["messages"][-1]["content"][:1500])

## 4. End-to-end LLM evaluation (needs API key)
`python scripts/eval_llm.py` answers the 57 test questions with Claude and scores citation validity, gold-page citation, language match and a Claude-as-judge correctness score against the reference answers. Results: `results/llm_metrics.json`.

In [ ]:
m = config.RESULTS / "llm_metrics.json"
print(m.read_text() if m.exists() else "Not run yet: set ANTHROPIC_API_KEY and run python scripts/eval_llm.py")